# LTX-Video 13B (FP8) on Colab L4

This notebook sets up the LTX-Video 13B model (FP8 quantized) on a Google Colab L4 instance. It includes:
1.  **Image-to-Video** generation.
2.  **TeaCache** integration for faster inference.
3.  **Modified Torch Commands** for performance tuning.
4.  **Gradio Interface** for easy interaction.

In [ ]:
#@title 1. Install Dependencies
#@markdown This step installs the necessary libraries and the Q8 Kernels for FP8 support. It may take a few minutes.

!git clone https://github.com/Lightricks/LTX-Video.git
%cd LTX-Video
!pip install -e .[inference]
!pip install gradio accelerate sentencepiece diffusers huggingface_hub

# Install Q8 Kernels for FP8 support (Required for 13B FP8 model)
!git clone https://github.com/Lightricks/LTXVideo-Q8-Kernels.git
!pip install -e LTXVideo-Q8-Kernels


In [ ]:
#@title 2. Import Libraries & Definitions
#@markdown Defines helper functions, patches TeaCache, and sets up the pipeline creation logic with FP8 support.

import os
import sys
import torch
import gradio as gr
import yaml
import random
import json
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import imageio
import tempfile
from safetensors import safe_open
from huggingface_hub import hf_hub_download
from diffusers.utils import logging
from typing import Optional, List, Union

# Add repo to path
sys.path.append(os.getcwd())

from ltx_video.models.autoencoders.causal_video_autoencoder import CausalVideoAutoencoder
from ltx_video.models.transformers.symmetric_patchifier import SymmetricPatchifier
from ltx_video.models.transformers.transformer3d import Transformer3DModel, Transformer3DModelOutput
from ltx_video.pipelines.pipeline_ltx_video import ConditioningItem, LTXVideoPipeline, LTXMultiScalePipeline
from ltx_video.schedulers.rf import RectifiedFlowScheduler
from ltx_video.models.autoencoders.latent_upsampler import LatentUpsampler
from ltx_video.utils.skip_layer_strategy import SkipLayerStrategy
import ltx_video.pipelines.crf_compressor as crf_compressor
from transformers import T5EncoderModel, T5Tokenizer, AutoModelForCausalLM, AutoProcessor, AutoTokenizer

# --- Helpers ---
def calculate_padding(source_height, source_width, target_height, target_width):
    pad_height = target_height - source_height
    pad_width = target_width - source_width
    pad_top = pad_height // 2
    pad_bottom = pad_height - pad_top
    pad_left = pad_width // 2
    pad_right = pad_width - pad_left
    return (pad_left, pad_right, pad_top, pad_bottom)

def load_image_to_tensor_with_resize_and_crop(image_input, target_height, target_width):
    if isinstance(image_input, str):
        image = Image.open(image_input).convert("RGB")
    else:
        image = image_input
    
    input_width, input_height = image.size
    aspect_ratio_target = target_width / target_height
    aspect_ratio_frame = input_width / input_height
    
    if aspect_ratio_frame > aspect_ratio_target:
        new_width = int(input_height * aspect_ratio_target)
        new_height = input_height
        x_start = (input_width - new_width) // 2
        y_start = 0
    else:
        new_width = input_width
        new_height = int(input_width / aspect_ratio_target)
        x_start = 0
        y_start = (input_height - new_height) // 2

    image = image.crop((x_start, y_start, x_start + new_width, y_start + new_height))
    image = image.resize((target_width, target_height))
    
    image = np.array(image)
    image = cv2.GaussianBlur(image, (3, 3), 0)
    frame_tensor = torch.from_numpy(image).float()
    frame_tensor = crf_compressor.compress(frame_tensor / 255.0) * 255.0
    frame_tensor = frame_tensor.permute(2, 0, 1)
    frame_tensor = (frame_tensor / 127.5) - 1.0
    return frame_tensor.unsqueeze(0).unsqueeze(2)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

# --- Pipeline Creation with FP8 Support ---
def create_transformer(ckpt_path, precision):
    if precision == "float8_e4m3fn":
        try:
            from q8_kernels.integration.patch_transformer import (
                patch_diffusers_transformer as patch_transformer_for_q8_kernels,
            )
            print("Loading FP8 Transformer with Q8 Kernels...")
            transformer = Transformer3DModel.from_pretrained(ckpt_path, dtype=torch.float8_e4m3fn)
            patch_transformer_for_q8_kernels(transformer)
            return transformer
        except ImportError:
            raise ValueError("Q8-Kernels not found. Please install LTXVideo-Q8-Kernels.")
    elif precision == "bfloat16":
        return Transformer3DModel.from_pretrained(ckpt_path).to(torch.bfloat16)
    else:
        return Transformer3DModel.from_pretrained(ckpt_path)

def custom_create_pipeline(ckpt_path, precision, text_encoder_path, device):
    ckpt_path = Path(ckpt_path)
    vae = CausalVideoAutoencoder.from_pretrained(ckpt_path).to(torch.bfloat16).to(device)
    transformer = create_transformer(ckpt_path, precision).to(device)
    scheduler = RectifiedFlowScheduler.from_pretrained(ckpt_path)
    text_encoder = T5EncoderModel.from_pretrained(text_encoder_path, subfolder="text_encoder").to(torch.bfloat16).to(device)
    tokenizer = T5Tokenizer.from_pretrained(text_encoder_path, subfolder="tokenizer")
    patchifier = SymmetricPatchifier(patch_size=1)
    
    pipe = LTXVideoPipeline(
        tokenizer=tokenizer, text_encoder=text_encoder, vae=vae, transformer=transformer,
        scheduler=scheduler, patchifier=patchifier,
        prompt_enhancer_image_caption_model=None, prompt_enhancer_image_caption_processor=None,
        prompt_enhancer_llm_model=None, prompt_enhancer_llm_tokenizer=None
    )
    return pipe

# --- TeaCache Integration ---
original_transformer_forward = Transformer3DModel.forward

def teacache_wrapper_forward(self, hidden_states: torch.Tensor, **kwargs):
    if not hasattr(self, "enable_teacache") or not self.enable_teacache:
        return original_transformer_forward(self, hidden_states=hidden_states, **kwargs)

    should_calc = True
    if self.cnt > 0 and self.cnt < self.num_steps - 1:
        if (hasattr(self, "previous_hidden_states") and
                self.previous_hidden_states is not None and
                self.previous_hidden_states.shape == hidden_states.shape):
            
            rel_l1_dist = ((hidden_states - self.previous_hidden_states).abs().mean() / self.previous_hidden_states.abs().mean()).cpu().item()
            self.accumulated_rel_l1_distance += rel_l1_dist

            if self.accumulated_rel_l1_distance < self.rel_l1_thresh:
                should_calc = False
            else:
                self.accumulated_rel_l1_distance = 0
        else:
            self.accumulated_rel_l1_distance = 0
    
    self.cnt += 1

    if not should_calc and hasattr(self, "previous_residual") and self.previous_residual is not None and self.previous_residual.shape == hidden_states.shape:
        return Transformer3DModelOutput(sample=self.previous_residual + hidden_states)
    else:
        self.previous_hidden_states = hidden_states.clone()
        output = original_transformer_forward(self, hidden_states=hidden_states, **kwargs)
        
        if isinstance(output, tuple):
            output_tensor = output[0]
        else:
            output_tensor = output.sample
            
        self.previous_residual = output_tensor - hidden_states
        return output

Transformer3DModel.forward = teacache_wrapper_forward
print("✅ TeaCache patched successfully.")

In [ ]:
#@title 3. Model Loading & Configuration
#@markdown Loads the 13B (Distilled, FP8) model. This configuration fits within the 24GB VRAM of an L4 GPU.

config_file_path = "configs/ltxv-13b-0.9.8-distilled-fp8.yaml"
with open(config_file_path, "r") as file:
    PIPELINE_CONFIG_YAML = yaml.safe_load(file)

LTX_REPO = "Lightricks/LTX-Video"
models_dir = "downloaded_models"
Path(models_dir).mkdir(parents=True, exist_ok=True)

print("Downloading models...")
# Download checkpoint
checkpoint_path = hf_hub_download(repo_id=LTX_REPO, filename=PIPELINE_CONFIG_YAML["checkpoint_path"], local_dir=models_dir)

# Create Pipeline
print("Initializing Pipeline...")
# Use CPU initially to save VRAM during load
pipeline_instance = custom_create_pipeline(
    ckpt_path=checkpoint_path,
    precision=PIPELINE_CONFIG_YAML["precision"], # float8_e4m3fn
    text_encoder_path=PIPELINE_CONFIG_YAML["text_encoder_model_name_or_path"],
    device="cpu" 
)

target_device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline_instance.to(target_device)
print(f"Pipeline loaded on {target_device}.")

In [ ]:
#@title 4. Generation Logic

def generate(image_input, prompt, negative_prompt, steps, guidance_scale, 
             enable_teacache, teacache_threshold, 
             enable_tf32, matmul_precision):
    
    if not image_input:
        raise gr.Error("Please upload an image.")

    # --- Modified Torch Commands ---
    torch.backends.cuda.matmul.allow_tf32 = enable_tf32
    torch.backends.cudnn.allow_tf32 = enable_tf32
    torch.set_float32_matmul_precision(matmul_precision)
    
    # --- TeaCache Config ---
    pipeline_instance.transformer.enable_teacache = enable_teacache
    pipeline_instance.transformer.rel_l1_thresh = teacache_threshold
    pipeline_instance.transformer.num_steps = steps
    pipeline_instance.transformer.cnt = 0
    pipeline_instance.transformer.previous_hidden_states = None
    pipeline_instance.transformer.previous_residual = None
    pipeline_instance.transformer.accumulated_rel_l1_distance = 0

    seed = random.randint(0, 2**32 - 1)
    seed_everything(seed)
    
    # Resolution setup
    height, width = 704, 1216
    height_padded = ((height - 1) // 32 + 1) * 32
    width_padded = ((width - 1) // 32 + 1) * 32
    padding = calculate_padding(height, width, height_padded, width_padded)
    
    # Load Image
    media_tensor = load_image_to_tensor_with_resize_and_crop(image_input, height, width)
    conditioning_items = [ConditioningItem(torch.nn.functional.pad(media_tensor, padding).to(target_device), 0, 1.0)]
    
    kwargs = {
        "prompt": prompt,
        "negative_prompt": negative_prompt,
        "height": height_padded,
        "width": width_padded,
        "num_frames": 121,
        "num_inference_steps": steps,
        "frame_rate": 24,
        "generator": torch.Generator(device=target_device).manual_seed(seed),
        "output_type": "pt",
        "conditioning_items": conditioning_items,
        "decode_timestep": 0.05,
        "decode_noise_scale": 0.025,
        "stochastic_sampling": False,
        "image_cond_noise_scale": 0.15,
        "is_video": True,
        "vae_per_channel_normalize": True,
        "mixed_precision": False,
        "offload_to_cpu": False,
        "enhance_prompt": False,
        "guidance_scale": guidance_scale,
        "skip_layer_strategy": SkipLayerStrategy.AttentionValues # Default strategy
    }

    print(f"Generating with Seed: {seed}, Steps: {steps}, Guidance: {guidance_scale}")
    with torch.no_grad():
        result = pipeline_instance(**kwargs).images

    # Process Output
    pad_l, pad_r, pad_t, pad_b = padding
    result = result[:, :, :kwargs["num_frames"], pad_t:(-pad_b or None), pad_l:(-pad_r or None)]
    
    video_np = (result[0].permute(1, 2, 3, 0).cpu().float().numpy() * 255).astype(np.uint8)
    output_path = tempfile.mktemp(suffix=".mp4")
    imageio.mimsave(output_path, video_np, fps=24, codec="libx264")
    
    return output_path

# --- Gradio UI ---
with gr.Blocks(title="LTX-Video 13B (FP8)") as demo:
    gr.Markdown("# LTX-Video 13B (FP8) Image-to-Video")
    gr.Markdown("Running on L4 Colab (24GB VRAM). Includes TeaCache & Optimization Controls.")
    
    with gr.Row():
        with gr.Column():
            image_in = gr.Image(label="Input Image", type="filepath")
            prompt_in = gr.Textbox(label="Prompt", value="Cinematic shot, subtle movement")
            neg_prompt_in = gr.Textbox(label="Negative Prompt", value="worst quality, blurry, distorted, low resolution")
            
            with gr.Accordion("Advanced Settings", open=False):
                steps = gr.Slider(label="Steps", minimum=10, maximum=50, value=30, step=1)
                guidance = gr.Slider(label="Guidance Scale", minimum=1, maximum=5, value=3.0, step=0.1)
                
            with gr.Accordion("Modified Torch Commands / Alternative Routes", open=True):
                teacache = gr.Checkbox(label="Enable TeaCache", value=True)
                teacache_thresh = gr.Slider(label="TeaCache Threshold", minimum=0.0, maximum=0.2, value=0.05)
                tf32 = gr.Checkbox(label="Allow TF32", value=True)
                precision = gr.Dropdown(label="MatMul Precision", choices=["medium", "high", "highest"], value="high")
                
            btn = gr.Button("Generate", variant="primary")
            
        with gr.Column():
            video_out = gr.Video(label="Output Video")

    btn.click(generate, inputs=[image_in, prompt_in, neg_prompt_in, steps, guidance, teacache, teacache_thresh, tf32, precision], outputs=video_out)

demo.launch(share=True, debug=True)
